In [ ]:
import pandas as pd

In [ ]:
#pip install kdepy
import loompy
import scvelo as scv
import scanpy as sc

In [ ]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation

from IPython.display import display, HTML



In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="numpy.dtype size changed")

import dynamo as dyn

dyn.configuration.set_figure_params('dynamo', background='white')
#dyn.pl.style(font_path='Arial')
dyn.get_all_dependencies_version()

%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.chdir('/mnt/d/KP/WT_PC_PN_PCN/2023_WTPCPNPCN/Dynamo/')

In [ ]:
adata= sc.read('WT_PC_PN_PCN_integrated.h5ad')
adata

In [ ]:
sc.pl.umap(adata, color=['batch'])

In [ ]:
adata_wtpcn = adata[adata.obs['batch'].isin(["WT","PCN"])]
adata_wtpcn

In [ ]:
adata = adata_wtpcn
adata

In [ ]:
from dynamo.preprocessing import Preprocessor

In [ ]:
celltype_key = "louvain"

In [ ]:
preprocessor = Preprocessor()
preprocessor.preprocess_adata(adata, recipe="monocle")

# Alternative
# preprocessor.config_monocle_recipe(adata)
# preprocessor.preprocess_adata_monocle(adata)

default_preprocessor_monocle_adata = adata # save for usage later

In [ ]:
dyn.tl.reduceDimension(adata, basis="pca")
dyn.pl.umap(adata, color=celltype_key)

In [ ]:
sc.pl.umap(adata, color=celltype_key,size=10, legend_loc='right margin')
sc.pl.umap(adata, color=celltype_key,size=10, legend_loc='on data')

In [ ]:
adata.layers.keys()

In [ ]:
cluster_map = {
    "0": "Org_2",
    "1": "Org_9",
    "2": "Org_7",
    "3": "Org_10",
    "4": "Org_8",
    "5": "Org_3",
    "6": "Org_1",
    "7": "Org_6",
    "8": "Org_11",
    "9": "Org_5",
    "10": "Org_4",
    "11": "Org_12"
}

adata.obs['cluster'] = adata.obs['louvain'].map(cluster_map)

In [ ]:
adata.obs.louvain

In [ ]:
adata.obs['cluster']

In [ ]:
sc.pl.umap(adata, color=['cluster'],size=10, legend_loc='on data')

In [ ]:
new_order = [
    "Org_1", "Org_2", "Org_3", "Org_4", "Org_5", "Org_6",
    "Org_7", "Org_8", "Org_9", "Org_10", "Org_11", "Org_12"
]

adata.obs['cluster'] = adata.obs['cluster'].astype('category')
adata.obs['cluster'] = adata.obs['cluster'].cat.reorder_categories(new_order, ordered=True)

In [ ]:
adata.obs['cluster'].cat.categories

In [ ]:
dyn.pl.umap(adata, color=['cluster'])

In [ ]:
adata

In [ ]:
# 2. Preprocessing
dyn.pp.recipe_monocle(adata)
# the author use dyn.pp.recipe_monocle(adata), which uses similar but generalized strategy from Monocle 3:
# converting ensemble gene names to gene official name and set them as .var_names if needed.
# making cell (.obs_names) and gene names (.var_names) unique.
# detecting your experiment type (conventional scRNA-seq or time-resolved metabolic labeling datasets) and normalize layers (the spliced and unspliced or metabolic labelled, and total mRNAs) and log1p transform the data
# calculating number of expressed genes (nGenes), total expression values (nCounts), percentage of total mitochondria gene values (pMito) for each cell and save them to .obs.
# calculating each cell’s cell cycle stage score.
# calculating new to total ratio (ntr) for each gene and cell.
# normalizes all datasets in different layers, do feature selection, log1p transformation and PCA dimension reduction.
# adding collapsed new, total and unspliced, spliced layers from the uu, ul, su, sl layers of a metabolic labeling experiment.
adata

In [ ]:
# 3. RNA velocity with parallelism
# 3 modes: deterministic, stochastic and negbin. Negbin is a novel method that relies on the negative binomial formulation of gene exrpession distribution at steady state.
dyn.tl.dynamics(adata, model='stochastic', cores=8)
# dyn.tl.dynamics() checks the experimental type automatically and learn the velocity for each feature gene, usually stochastic or negative binomial method for conventional scRNA-seq; kinetic, degradation or one-shot models for metabolic labeling based scRNA-seq.
adata

In [ ]:
dyn.tl.reduceDimension(adata, basis='umap', n_pca_components=30, n_neighbors=30, reduction_method='umap', cores=8)
# if basis='pca', it uses HDBSCAN for clustering, you can use basis='umap'
adata

In [ ]:
adata.obs['cluster'].dtype

In [ ]:
dyn.pl.umap(adata, color='cluster')

In [ ]:
# if you observe undesired backward vector flow. You can evaluate the confidence of gene-wise velocity via
    ## dyn.tl.gene_wise_confidence(adata, group='group', lineage_dict={'Progenitor': ['terminal_cell_state']})
    # lineage_dict is a dictionary indicates the progenitor group, the values is a list including the possible terminal cell groups

In [ ]:
# check phase diagram
dyn.pl.phase_portraits(adata, genes=adata.var_names[adata.var.use_for_dynamics][:4], basis='umap', color='cluster')
# pvalb1 gene has a strange phase diagram with a few cells have high spliced expression values but extremely low unspliced expression values.
# This kind of phase space may points to inproper intron capture of those genes during the library prepartion or sequencing and they should never be used for velocity projection and vector field analysis.

In [ ]:
# 4. Velocity projection
# The projection calculates a transition matrix between cells for local averaging of velocity vectors into low dimension.
dyn.tl.cell_velocities(adata, basis='umap', n_neighbors = 30, method='pearson', other_kernels_dict={'transform': 'sqrt'})
# There are 3 methods to calculate the transition matrix, either kmc, cosine, pearson. kmc is our new approach to learn the transition matrix via diffusion approximation or an Itô kernel. cosine or pearson are the methods used in the original velocyto or the scvelo paper.
adata

In [ ]:
# check the confidence of cell-wise velocity to understand how reliable the recovered velocity is across cells
dyn.tl.cell_wise_confidence(adata, method='jaccard')

In [ ]:
adata.obs['cluster']

In [ ]:
## dyn.tl.confident_cell_velocities(adata, group='Cell_type', lineage_dict={'Proliferating Progenitor': ['Schwann Cell', 'Melanophore']}, basis='umap', confidence_threshold=0.85)
#adata

In [ ]:
dyn.pl.cell_wise_vectors(adata, basis='umap', color=['cluster'], show_legend='on data', quiver_length=6, quiver_size=6, pointsize=0.1, show_arrowed_spines=False)

In [ ]:
dyn.pl.streamline_plot(adata, color=['cluster','batch'], basis='umap', show_legend='on data', show_arrowed_spines=True)

In [ ]:
dyn.pl.streamline_plot(adata, color=['cluster','batch'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
# 5. Reconstruct vector field
dyn.vf.VectorField(adata, basis='umap', M=1000, pot_curl_div=True)
adata

In [ ]:
dyn.pl.plot_energy(adata, basis='umap')
adata

In [ ]:
# 6. Characterize vector field topology
# characterize the topology of the full vector field
dyn.pl.topography(adata, basis='umap', color=['ntr', 'cluster'], background='white', streamline_color='black', show_legend='on data', frontier=True)
# the fixed points corresponds to terminal cell types or progenitors: black is absorbing fixed points; red is emitting fixed points; blue is unstable fixed points.
adata

In [ ]:
# like pseudotime
dyn.pl.umap(adata, color='umap_ddhodge_potential', frontier=True)

In [ ]:
dyn.pl.scatters(adata, x=np.repeat('umap_ddhodge_potential', 4), y=['Mki67','Trp63','Sox2','Krt13'],
                color='louvain', layer='X_spliced', background='white', ncols=3, pointsize=0.25, alpha=0.8)

In [ ]:
# 7. Beyond RNA velocity
dyn.tl.cell_velocities(adata, basis='pca')
dyn.vf.VectorField(adata, basis='pca')
dyn.vf.speed(adata, basis='pca')
dyn.vf.curl(adata, basis='umap')
dyn.vf.divergence(adata, basis='pca')
dyn.vf.acceleration(adata, basis='pca')
dyn.vf.curvature(adata, basis='pca')

In [ ]:
# 8. Integrative analysis
dyn.pl.cell_wise_vectors(adata, color='speed_pca', pointsize=0.5, alpha = 0.7, quiver_length=6, quiver_size=6, save_show_or_return='return')
dyn.pl.grid_vectors(adata, color='divergence_pca', quiver_length=12, quiver_size=12, save_show_or_return='return')
dyn.pl.streamline_plot(adata, color='acceleration_pca', save_show_or_return='return')
dyn.pl.streamline_plot(adata, color='curvature_pca', save_show_or_return='return')

In [ ]:
adata.write('WT_PCN_only_intermediate_2.h5ad', compression='gzip')

# Check 

In [ ]:
# 9. Animate fate transition
progenitor = adata.obs_names[adata.obs.louvain.isin(['Org_9','Org_11','Org_12'])]    # select some progenitor cells as the initial cell states
len(progenitor)

In [ ]:
# predict the cell fate trajectory via integrating with the vector field function, starting from the initial cell states
dyn.pd.fate(adata, init_cells=progenitor, basis='umap', direction='forward', interpolation_num=100, average=False, inverse_transform=False, cores=40)
adata

In [ ]:
# use the topography plot as the background, the animated components from each frame will be plotted on its top
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata, color='louvain', ax=ax, save_show_or_return='return')

In [ ]:
instance = dyn.mv.StreamFuncAnim(adata=adata, color='louvain', ax=ax)
matplotlib.rcParams['animation.embed_limit'] = 2**128    # Ensure all frames will be embedded.
anim = animation.FuncAnimation(instance.fig, instance.update, init_func=instance.init_background, frames=np.arange(100), interval=100, blit=True)
HTML(anim.to_jshtml())    # embedding the movie to jupyter notebook

In [ ]:
# or save the animation as an gif file
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata, color='louvain', ax=ax, save_show_or_return='return')
dyn.mv.animate_fates(adata, color='louvain', basis='umap', n_steps=100, fig=fig, ax=ax, save_show_or_return='save', logspace=True, max_time=None)

In [ ]:
dyn.cleanup(adata)
del adata.uns['velocyto_SVR']
del adata.uns['VecFld_umap']
del adata.uns['fate_umap']
adata

In [ ]:
adata.write('only_normal-CIS_integrated_dynamo.h5ad')